# Análisis Exploratorio de Datos (EDA) — World Cup Insights

**Curso:** BD-143 Programación II — II Cuatrimestre 2026
**Proyecto II:** Sistema de análisis de la Copa Mundial de la FIFA (1930–2026)

## Objetivo
Ingerir, limpiar y explorar el histórico de partidos de Copa Mundial para
identificar patrones de rendimiento, y dejar el dataset procesado listo para
la etapa de visualización.

Este notebook invoca las clases `CargadorDatos`, `ProcesadorEDA` y `GestorPartidos`.

## 1. Configuración e importación de las clases del proyecto

In [1]:
import sys
import os

import pandas as pd

# Se agrega la raíz del proyecto al path para poder importar el paquete src.
sys.path.append(os.path.abspath('..'))

from src.ingesta.cargador import CargadorDatos
from src.gestor.gestor import GestorPartidos
from src.eda.procesador import ProcesadorEDA
from src.helpers.utilidades import Utilidades

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('Clases importadas correctamente.')

Clases importadas correctamente.


## 2. Ingesta de datos

`CargadorDatos` descarga el CSV público de resultados internacionales y filtra
únicamente los partidos cuyo torneo es *FIFA World Cup*.

Se usa `obtener_datos()`, que reutiliza el archivo local si ya fue descargado.
Así el notebook corre aunque no haya conexión a internet.

In [2]:
cargador = CargadorDatos(
    ruta_raw=Utilidades.ruta_proyecto('data', 'raw', 'partidos-mundial.csv'),
    ruta_processed=Utilidades.ruta_proyecto('data', 'processed', 'partidos-mundial-procesado.csv'),
)

df_raw = cargador.obtener_datos()
print(f'Partidos de Copa Mundial: {len(df_raw)}')
df_raw.head()

[OK] Datos leidos desde disco: C:\Users\Daniel Nájera\Documents\GitHub\Proyecto2Programacion\world_cup_insights\data\raw\partidos-mundial.csv (1068 partidos)
Partidos de Copa Mundial: 1068


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1930-07-13,Belgium,United States,0,3,FIFA World Cup,Montevideo,Uruguay,True
1,1930-07-13,France,Mexico,4,1,FIFA World Cup,Montevideo,Uruguay,True
2,1930-07-14,Brazil,Yugoslavia,1,2,FIFA World Cup,Montevideo,Uruguay,True
3,1930-07-14,Peru,Romania,1,3,FIFA World Cup,Montevideo,Uruguay,True
4,1930-07-15,Argentina,France,1,0,FIFA World Cup,Montevideo,Uruguay,True


### 2.1 Estructura del dataset original

In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        1068 non-null   str  
 1   home_team   1068 non-null   str  
 2   away_team   1068 non-null   str  
 3   home_score  1068 non-null   int64
 4   away_score  1068 non-null   int64
 5   tournament  1068 non-null   str  
 6   city        1068 non-null   str  
 7   country     1068 non-null   str  
 8   neutral     1068 non-null   bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 125.5 KB


### 2.2 Validación de la calidad de los datos

Antes de limpiar conviene medir: `validar_datos()` reporta duplicados, nulos,
marcadores negativos y el rango de fechas, sin modificar nada.

In [4]:
reporte = cargador.validar_datos(df_raw)
print(Utilidades.resumen_en_texto(
    {k: v for k, v in reporte.items() if k != 'nulos_por_columna'},
    'Reporte de validación',
))

print('\nNulos por columna:')
print(pd.Series(reporte['nulos_por_columna']))

Reporte de validación
---------------------
total_filas          : 1068
filas_duplicadas     : 0
marcadores_negativos : 0
fechas_invalidas     : 0
rango_fechas         : ('1930-07-13', '2026-07-19')

Nulos por columna:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64


## 3. Limpieza de datos

`ProcesadorEDA.limpieza_datos()` convierte la fecha a `datetime`, descarta
partidos sin marcador, elimina duplicados exactos, normaliza los nombres de
equipos y sedes, y pasa los marcadores a entero.

In [5]:
procesador = ProcesadorEDA(df_raw)
df_limpio = procesador.limpieza_datos()

print(f'Filas antes de limpiar : {len(df_raw)}')
print(f'Filas después de limpiar: {len(df_limpio)}')
df_limpio.dtypes

Filas antes de limpiar : 1068
Filas después de limpiar: 1068


date          datetime64[us]
home_team                str
away_team                str
home_score             int64
away_score             int64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

## 4. Columnas derivadas

Se agregan las columnas que pide el enunciado:

| Columna | Significado |
|---|---|
| `year` | Edición del Mundial |
| `total_goals` | Goles totales del partido |
| `goal_diff` | Diferencia de goles (local − visitante) |
| `winner` | Local / Visitante / Empate |
| `decada` | Década, para agrupaciones |
| `es_empate` | Booleano de apoyo |

In [6]:
df = procesador.crear_columnas_derivadas()
df[['date', 'home_team', 'home_score', 'away_score', 'away_team',
    'year', 'total_goals', 'goal_diff', 'winner']].head(10)

,date,home_team,home_score,away_score,away_team,year,total_goals,goal_diff,winner
0,1930-07-13,Belgium,0,3,United States,1930,3,-3,Visitante
1,1930-07-13,France,4,1,Mexico,1930,5,3,Local
2,1930-07-14,Brazil,1,2,Yugoslavia,1930,3,-1,Visitante
3,1930-07-14,Peru,1,3,Romania,1930,4,-2,Visitante
4,1930-07-15,Argentina,1,0,France,1930,1,1,Local
5,1930-07-16,Chile,3,0,Mexico,1930,3,3,Local
6,1930-07-17,Bolivia,0,4,Yugoslavia,1930,4,-4,Visitante
7,1930-07-17,Paraguay,0,3,United States,1930,3,-3,Visitante
8,1930-07-18,Uruguay,1,0,Peru,1930,1,1,Local
9,1930-07-19,Argentina,6,3,Mexico,1930,9,3,Local


## 5. Resumen descriptivo

In [7]:
procesador.resumen_descriptivo()

,home_score,away_score,total_goals,goal_diff
count,1068.000000,1068.000000,1068.000000,1068.000000
mean,1.583333,1.251873,2.835206,0.331461
std,1.494000,1.292051,1.920880,2.028074
min,0.000000,0.000000,0.000000,-8.000000
25%,0.750000,0.000000,1.000000,-1.000000
50%,1.000000,1.000000,3.000000,0.000000
75%,2.000000,2.000000,4.000000,1.000000
max,10.000000,8.000000,12.000000,9.000000


**Lectura:** el promedio de goles por partido ronda los 2.8, con una mediana de
3. El máximo histórico es de 12 goles en un mismo partido, muy por encima del
tercer cuartil, lo que anticipa la presencia de valores atípicos.

## 6. Matriz de correlación

In [8]:
procesador.matriz_correlacion()

,home_score,away_score,total_goals,goal_diff
home_score,1.000000,-0.054823,0.740893,0.771586
away_score,-0.054823,1.000000,0.629995,-0.677469
total_goals,0.740893,0.629995,1.000000,0.144426
goal_diff,0.771586,-0.677469,0.144426,1.000000


**Lectura:** `home_score` y `away_score` tienen correlación *negativa*
(≈ −0.05 a −0.06). Los partidos de Mundial tienden a ser desequilibrados:
cuando un equipo golea, el rival rara vez responde con goles propios.

## 7. Valores nulos tras la limpieza

In [9]:
procesador.reporte_nulos()

,nulos,porcentaje
date,0,0.0
home_team,0,0.0
away_team,0,0.0
home_score,0,0.0
away_score,0,0.0
tournament,0,0.0
city,0,0.0
country,0,0.0
neutral,0,0.0
year,0,0.0


## 8. Detección de outliers

Se aplica el método del rango intercuartílico (IQR). En este dataset los
outliers **no son errores de captura**: son las goleadas históricas, es decir
los partidos más interesantes del torneo.

In [10]:
outliers = procesador.detectar_outliers('total_goals')

print(Utilidades.resumen_en_texto({
    'Q1': outliers['q1'],
    'Q3': outliers['q3'],
    'IQR': outliers['iqr'],
    'Límite superior': outliers['limite_superior'],
    'Partidos atípicos': outliers['cantidad_outliers'],
}, 'Outliers en total_goals'))

outliers['outliers'][['date', 'home_team', 'home_score',
                      'away_score', 'away_team', 'total_goals']].head(10)

Outliers en total_goals
-----------------------
Q1                : 1.00
Q3                : 4.00
IQR               : 3.00
Límite superior   : 8.50
Partidos atípicos : 11


,date,home_team,home_score,away_score,away_team,total_goals
94,1954-06-26,Switzerland,5,7,Austria,12
88,1954-06-20,Germany,3,8,Hungary,11
36,1938-06-05,Brazil,6,5,Poland,11
311,1982-06-15,Hungary,10,1,El Salvador,11
105,1958-06-08,France,7,3,Paraguay,10
1066,2026-07-18,France,4,6,England,10
9,1930-07-19,Argentina,6,3,Mexico,9
81,1954-06-17,Hungary,9,0,South Korea,9
91,1954-06-23,Germany,7,2,Turkey,9
243,1974-06-18,Yugoslavia,9,0,DR Congo,9


## 9. Agrupaciones: cada Mundial y cada década

In [11]:
resumen_ediciones = procesador.agrupar_por_edicion()
resumen_ediciones

,partidos,goles_totales,promedio_goles,max_goles_partido,empates,porcentaje_empates
year,,,,,,
1930,18,70,3.89,9,0,0.0
1934,17,70,4.12,8,1,5.9
1938,18,84,4.67,11,3,16.7
1950,22,88,4.00,8,3,13.6
1954,26,140,5.38,12,2,7.7
1958,35,126,3.60,10,10,28.6
1962,32,89,2.78,8,5,15.6
1966,32,89,2.78,8,5,15.6
1970,32,95,2.97,7,5,15.6


> **Cuidado al comparar ediciones:** el torneo pasó de 18 partidos en 1930 a
> **104 en 2026** (formato de 48 equipos). Por eso los *goles totales* no son
> comparables entre ediciones y hay que usar el **promedio por partido**.

In [12]:
procesador.agrupar_por_decada()

,partidos,promedio_goles
decada,,
1930,53,4.23
1950,83,4.27
1960,64,2.78
1970,108,2.72
1980,104,2.67
1990,168,2.54
2000,128,2.41
2010,192,2.53
2020,168,2.86


## 10. Distribución de resultados

In [13]:
procesador.distribucion_resultados()

,cantidad,porcentaje
winner,,
Local,488,45.69
Visitante,342,32.02
Empate,238,22.28


## 11. Consultas con `GestorPartidos`

Todos estos métodos son de **solo lectura**: nunca modifican el DataFrame.

In [14]:
gestor = GestorPartidos(df)

print('Ediciones disponibles:', gestor.get_ediciones())
print('Total de selecciones :', len(gestor.get_equipos()))
print('Países sede          :', len(gestor.get_sedes()))

Ediciones disponibles: [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022, 2026]
Total de selecciones : 86
Países sede          : 19


### 11.1 Consultas por equipo, año y sede

In [15]:
print('Partidos de Costa Rica :', len(gestor.get_por_equipo('Costa Rica')))
print('Partidos del Mundial 2022:', len(gestor.get_por_anio(2022)))
print('Partidos jugados en Brasil:', len(gestor.get_por_sede('Brazil')))

gestor.get_por_equipo('Costa Rica')[
    ['date', 'home_team', 'home_score', 'away_score', 'away_team', 'year']
].head()

Partidos de Costa Rica : 21


Partidos del Mundial 2022: 64
Partidos jugados en Brasil: 86


,date,home_team,home_score,away_score,away_team,year
419,1990-06-11,Costa Rica,1,0,Scotland,1990
430,1990-06-16,Brazil,1,0,Costa Rica,1990
443,1990-06-20,Sweden,1,2,Costa Rica,1990
449,1990-06-23,Czechoslovakia,4,1,Costa Rica,1990
591,2002-06-04,China,0,2,Costa Rica,2002


### 11.2 Ficha histórica de una selección

In [16]:
print(Utilidades.resumen_en_texto(gestor.resumen_equipo('Brazil'),
                                  'Brasil en Copas del Mundo'))

Brasil en Copas del Mundo
-------------------------
equipo            : Brazil
partidos_jugados  : 119
ganados           : 79
empatados         : 20
perdidos          : 20
goles_favor       : 247
goles_contra      : 112
diferencia_goles  : 135
mundiales_jugados : 23


### 11.3 Duelos directos

In [17]:
gestor.get_enfrentamientos('Argentina', 'Germany')[
    ['date', 'home_team', 'home_score', 'away_score', 'away_team', 'year']
]

,date,home_team,home_score,away_score,away_team,year
101,1958-06-08,Argentina,1,3,Germany,1958
180,1966-07-16,Argentina,0,0,Germany,1966
411,1986-06-29,Argentina,3,2,Germany,1986
463,1990-07-08,Germany,1,0,Argentina,1990
700,2006-06-30,Germany,1,1,Argentina,2006
766,2010-07-03,Argentina,0,4,Germany,2010
835,2014-07-13,Germany,1,0,Argentina,2014


### 11.4 La ventaja de local: el hallazgo principal

El porcentaje global de victorias del local es poco impresionante. Pero al
separar los partidos según haya localía real o cancha neutral, aparece el
verdadero efecto.

In [18]:
print('Global:', gestor.ventaja_local())
print()

for escenario, datos in gestor.ventaja_local_por_neutralidad().items():
    print(f"{escenario:>16}: {datos['porcentaje']:.1f}% "
          f"({datos['victorias_local']} de {datos['total_partidos']} partidos)")

Global: {'total_partidos': 1068, 'victorias_local': 488, 'porcentaje': 45.69288389513109}

    localia_real: 61.2% (82 de 134 partidos)
  cancha_neutral: 43.5% (406 de 934 partidos)


**Interpretación.** El promedio global (≈ 45.7 %) sugiere que ser local casi no
importa. Es un promedio engañoso: cuando existe localía real el equipo de casa
gana **61.2 %** de los partidos, contra **43.5 %** en cancha neutral.

Como el 87 % de los partidos de Mundial se juegan en cancha neutral, el grupo
mayoritario arrastra el promedio hacia abajo y esconde el efecto. Es un caso
clásico de **paradoja de Simpson**.

### 11.5 ¿El país sede gana más?

In [19]:
gestor.rendimiento_pais_sede().head(10)

,sede,partidos_en_casa,ganados,porcentaje_victorias
0,Uruguay,4,4,100.0
1,England,6,5,83.3
2,Italy,12,10,83.3
3,Germany,14,11,78.6
4,France,9,7,77.8
5,Argentina,7,5,71.4
6,Chile,6,4,66.7
7,Sweden,6,4,66.7
8,Mexico,14,9,64.3
9,Brazil,13,7,53.8


### 11.6 Tabla histórica y partidos más goleados

In [20]:
gestor.tabla_historica().head(10)

,equipo,partidos_jugados,ganados,empatados,perdidos,goles_favor,goles_contra,diferencia_goles,mundiales_jugados
0,Brazil,119,79,20,20,247,112,135,23
1,Germany,116,70,22,24,243,135,108,21
2,Argentina,96,54,17,25,171,109,62,19
3,France,81,45,14,22,156,95,61,17
4,Italy,83,45,21,17,128,77,51,18
5,Netherlands,59,32,16,11,107,57,50,12
6,Spain,75,38,18,19,122,76,46,17
7,England,82,38,23,21,124,80,44,17
8,Hungary,32,15,3,14,87,57,30,9
9,Portugal,40,19,8,13,69,44,25,9


In [21]:
gestor.partidos_mas_goleados(10)

,date,home_team,home_score,away_score,away_team,total_goals
0,1954-06-26,Switzerland,5,7,Austria,12
1,1938-06-05,Brazil,6,5,Poland,11
2,1954-06-20,Germany,3,8,Hungary,11
3,1982-06-15,Hungary,10,1,El Salvador,11
4,1958-06-08,France,7,3,Paraguay,10
5,2026-07-18,France,4,6,England,10
6,1930-07-19,Argentina,6,3,Mexico,9
7,1954-06-17,Hungary,9,0,South Korea,9
8,1954-06-23,Germany,7,2,Turkey,9
9,1958-06-28,France,6,3,Germany,9


## 12. Persistencia del dataset procesado

Se guarda el resultado en `data/processed/`, en CSV y en JSON.
Este archivo es el **entregable 7** del enunciado.

In [22]:
cargador.guardar_procesado(df)
cargador.guardar_procesado_json()

[OK] Dataset procesado guardado exitosamente en: C:\Users\Daniel Nájera\Documents\GitHub\Proyecto2Programacion\world_cup_insights\data\processed\partidos-mundial-procesado.csv
[OK] Dataset procesado guardado en JSON: C:\Users\Daniel Nájera\Documents\GitHub\Proyecto2Programacion\world_cup_insights\data\processed\partidos-mundial-procesado.json


'C:\\Users\\Daniel Nájera\\Documents\\GitHub\\Proyecto2Programacion\\world_cup_insights\\data\\processed\\partidos-mundial-procesado.json'

## 13. Conclusiones del EDA

1. **La localía sí pesa, pero el promedio global lo esconde.** 61.2 % con
   localía real contra 43.5 % en cancha neutral: una diferencia de casi 18
   puntos que el promedio de 45.7 % oculta (paradoja de Simpson).

2. **El fútbol de Mundial se volvió más cerrado.** Del pico de 5.38 goles por
   partido en 1954 se bajó a 2.21 en 1990, y desde entonces el promedio se
   estabilizó cerca de 2.6.

3. **Los países sede rinden mejor en casa:** Uruguay (100 %), Inglaterra e
   Italia (83 %), Alemania (79 %).

4. **Brasil domina el histórico** con +135 de diferencia de goles, seguido de
   Alemania (+108) y Argentina (+62).

5. **Los outliers no son errores:** son las goleadas históricas, encabezadas
   por el Suiza 5–7 Austria de 1954, con 12 goles.

En el notebook `02_Visualizacion.ipynb` estas conclusiones se traducen a gráficos.